# STLab02 - Requirement-Based Automated Test Execution & Reporting
**Môn học:** Software Testing  
**Bài tập:** STLab02 – Test Case Execution   
**Case Study:** EduCommerce Checkout System  
**Tên:** Lê Thị Ngọc Anh | **MSSV:** 23110131  
**GitHub Repository:** [https://github.com/ngocanh1002/STLab02_EduCommerce_Testing](https://github.com/ngocanh1002/STLab02_EduCommerce_Testing)

---
## 0. MỤC TIÊU VÀ PHẠM VI BÀI LÀM
1. Chọn 05 Requirements mới: **REQ-02 (Cart), REQ-04 (Auth), REQ-07 (Promotion), REQ-09 (Shipping), REQ-10 (Payment)**.

2. Thiết kế Ma trận Traceability: **5 REQs → 12 Conditions → 12 Scenarios → 19 Test Cases** (10 ca Negative/Boundary).

3. Chuẩn hóa bộ dữ liệu kiểm thử theo 15 cột bắt buộc (1.5 Requirement).

4. Tự động hóa kiểm thử bằng Python trên Colab, xuất báo cáo CSV/Excel/HTML và Gradio Dashboard.

5. **Kết quả kiểm thử thực tế:** 18/19 PASS, 1 FAIL (94.7% Pass Rate). Phát hiện Defect mức độ Critical **DEF-PAY-01** (Rò rỉ tồn kho khi cổng thanh toán Online từ chối giao dịch).


In [24]:
from pathlib import Path
print("Python runtime ready")
print("Working directory:", Path.cwd())

Python runtime ready
Working directory: g:\Download\testSoftware\Lab02


---
## PHẦN 1 - TẠO MÃ NGUỒN ỨNG DỤNG EDUCOMMERCE (`ecommerce_app.py`)

In [25]:
%%writefile ecommerce_app.py
"""EduCommerce: in-memory application for practicing checkout and automated testing."""
from dataclasses import dataclass, field
from decimal import Decimal, ROUND_HALF_UP
from typing import Optional

D=lambda x: Decimal(str(x))

def money(x): return D(x).quantize(D('1'), rounding=ROUND_HALF_UP)

@dataclass
class Product:
    sku:str; title:str; price:Decimal; stock:int; active:bool=True
@dataclass
class Customer:
    customer_id:str; authenticated:bool=True; used_codes:set=field(default_factory=set)
@dataclass
class CartLine:
    sku:str; quantity:int
@dataclass
class Voucher:
    code:str; percent:int; min_subtotal:Decimal; cap:Decimal; active:bool=True
@dataclass
class Order:
    order_id:str; customer_id:str; total:Decimal; status:str; lines:list; discount:Decimal; shipping:Decimal; payment_method:str

class Catalog:
    def __init__(self):
        self.products={'A':Product('A','Wireless Mouse',D(200000),20),
                       'B':Product('B','Mechanical Keyboard',D(300000),15),
                       'C':Product('C','USB-C Hub',D(150000),12)}
    def get(self,sku):
        p=self.products.get(sku)
        if not p or not p.active: raise ValueError('PRODUCT_UNAVAILABLE')
        return p

class Cart:
    def __init__(self,catalog): self.catalog=catalog; self.lines=[]
    def add(self,sku,quantity):
        self.catalog.get(sku)
        if not isinstance(quantity,int) or quantity<1: raise ValueError('INVALID_QUANTITY')
        existing=next((x for x in self.lines if x.sku==sku),None)
        if existing: existing.quantity+=quantity
        else: self.lines.append(CartLine(sku,quantity))
    def subtotal(self): return sum((self.catalog.get(x.sku).price*x.quantity for x in self.lines),D(0))

class Inventory:
    def __init__(self,catalog): self.catalog=catalog
    def reserve(self,lines):
        for line in lines:
            if self.catalog.get(line.sku).stock < line.quantity: raise ValueError('OUT_OF_STOCK')
        for line in lines: self.catalog.get(line.sku).stock-=line.quantity
    def release(self,lines):
        for line in lines: self.catalog.get(line.sku).stock+=line.quantity

class Promotions:
    def __init__(self): self.codes={'SAVE10':Voucher('SAVE10',10,D(500000),D(100000))}
    def discount(self, code, subtotal, customer):
        if not code: return D(0)
        v=self.codes.get(code)
        if not v or not v.active: raise ValueError('INVALID_VOUCHER')
        if not customer.authenticated: raise ValueError('LOGIN_REQUIRED')
        if code in customer.used_codes: raise ValueError('VOUCHER_USED')
        # Validate minimum subtotal for voucher eligibility.
        if subtotal <= v.min_subtotal: raise ValueError('MIN_SUBTOTAL_NOT_MET')
        return min(money(subtotal*v.percent/D(100)),v.cap)

class Shipping:
    def calculate(self,subtotal,area):
        if area not in ('inner','outer'): raise ValueError('INVALID_AREA')
        return D(0) if area=='inner' and subtotal>=D(500000) else (D(30000) if area=='inner' else D(50000))

class Payments:
    def __init__(self): self.charges=[]
    def collect(self,amount,method,approved,key):
        if method=='COD': return 'PENDING_COD'
        if method!='ONLINE': raise ValueError('INVALID_PAYMENT_METHOD')
        if not approved: raise ValueError('PAYMENT_DECLINED')
        self.charges.append({'key':key,'amount':amount})
        return 'PAID'

class Checkout:
    def __init__(self,catalog=None):
        self.catalog=catalog or Catalog(); self.inventory=Inventory(self.catalog)
        self.promotions=Promotions(); self.shipping=Shipping(); self.payments=Payments()
        self.orders={}; self.by_key={}; self.counter=0
    def place(self,customer,cart,area='inner',voucher=None,method='COD',approved=True,key='REQ-001'):
        if not customer.authenticated: raise ValueError('LOGIN_REQUIRED')
        if not cart.lines: raise ValueError('EMPTY_CART')
        subtotal=cart.subtotal()
        discount=self.promotions.discount(voucher,subtotal,customer)
        shipping=self.shipping.calculate(subtotal,area)
        total=max(D(0),subtotal-discount+shipping)
        # Reserve stock during the checkout transaction.
        self.inventory.reserve(cart.lines)
        if key in self.by_key: return self.orders[self.by_key[key]]
        # Process the online payment outcome.
        payment_state=self.payments.collect(total,method,approved,key)
        self.counter+=1
        order=Order(f'ORD-{self.counter:04d}',customer.customer_id,total,payment_state,
                    [(x.sku,x.quantity) for x in cart.lines],discount,shipping,method)
        self.orders[order.order_id]=order; self.by_key[key]=order.order_id
        if voucher: customer.used_codes.add(voucher)
        return order

def fixture(subtotal_case='600k'):
    service=Checkout(); customer=Customer('U_NEW'); cart=Cart(service.catalog)
    if subtotal_case=='500k': cart.add('A',1); cart.add('B',1)
    elif subtotal_case=='499999':
        cart.add('A',1); cart.add('B',1); service.catalog.products['B'].price=D(299999)
    elif subtotal_case=='1000k': cart.add('A',5)
    elif subtotal_case=='1500k': cart.add('B',5)
    elif subtotal_case=='600k': cart.add('A',3)
    elif subtotal_case=='300k': cart.add('B',1)
    else: raise ValueError('UNKNOWN_FIXTURE')
    return service,customer,cart

Overwriting ecommerce_app.py


### Kiểm tra nạp module `ecommerce_app`

In [26]:
import ecommerce_app
import importlib
importlib.reload(ecommerce_app)
print("ecommerce_app module reloaded successfully!")

ecommerce_app module reloaded successfully!


---
## PHẦN 2 - TEST CASES DATA-DRIVEN (`STLab02_TestCases.csv`)
Theo Mục 1.5 của file Requirement, cấu trúc 15 cột bắt buộc gồm:
`test_case_id, requirement_id, condition_id, scenario_id, title, priority, precondition, fixture_key, input_json, expected_type, expected_json, execution_status, actual_result, error_message, evidence_reference`

Dưới đây là 19 Test Cases thiết kế cho 5 REQs mới đã chọn:

In [27]:
%%writefile STLab02_TestCases.csv
test_case_id,requirement_id,condition_id,scenario_id,title,priority,precondition,fixture_key,input_json,expected_type,expected_json,execution_status,actual_result,error_message,evidence_reference
TC-CRT-001,REQ-02,COND-CRT-01,TS-CRT-VALID,Thêm sản phẩm mới hợp lệ vào giỏ hàng,P1,"Cart rỗng, catalog có sẵn sản phẩm A",clean_cart,"{""actions"": [{""sku"": ""A"", ""quantity"": 2}]}",state,"{""total_quantity"": 2, ""lines_count"": 1, ""subtotal"": 400000}",PENDING,,,Cart.add()
TC-CRT-002,REQ-02,COND-CRT-01,TS-CRT-VALID,Cộng dồn số lượng khi thêm cùng SKU nhiều lần,P1,"Cart đã có SKU A qty=2, thêm tiếp SKU A qty=3",clean_cart,"{""actions"": [{""sku"": ""A"", ""quantity"": 2}, {""sku"": ""A"", ""quantity"": 3}]}",state,"{""total_quantity"": 5, ""lines_count"": 1, ""subtotal"": 1000000}",PENDING,,,Cart.add() aggregation
TC-CRT-003,REQ-02,COND-CRT-02,TS-CRT-INVALID,Thêm sản phẩm với số lượng bằng 0 (biên không hợp lệ),P0,"Cart rỗng, sản phẩm A active",clean_cart,"{""actions"": [{""sku"": ""A"", ""quantity"": 0}]}",error,"{""error"": ""INVALID_QUANTITY""}",PENDING,,,Cart.add() zero validation
TC-CRT-004,REQ-02,COND-CRT-02,TS-CRT-INVALID,Thêm sản phẩm với số lượng âm (negative),P1,"Cart rỗng, sản phẩm A active",clean_cart,"{""actions"": [{""sku"": ""A"", ""quantity"": -1}]}",error,"{""error"": ""INVALID_QUANTITY""}",PENDING,,,Cart.add() negative validation
TC-AUT-001,REQ-04,COND-AUT-01,TS-AUT-LOGIN,Khách hàng đã đăng nhập thực hiện checkout thành công,P1,"Customer authenticated=True, giỏ hàng 600k",600k,"{""method"": ""COD""}",order_success,"{""status"": ""PENDING_COD""}",PENDING,,,Checkout.place() auth=True
TC-AUT-002,REQ-04,COND-AUT-02,TS-AUT-GUEST,Khách hàng chưa đăng nhập bị từ chối checkout,P0,"Customer authenticated=False, giỏ hàng 600k",unauth_user,"{""method"": ""COD""}",error,"{""error"": ""LOGIN_REQUIRED""}",PENDING,,,Checkout.place() auth=False
TC-AUT-003,REQ-04,COND-AUT-02,TS-AUT-GUEST,Khách chưa đăng nhập không tạo Order trong hệ thống,P1,"Customer authenticated=False, giỏ hàng 600k",unauth_user,"{""method"": ""ONLINE"", ""approved"": true}",error,"{""error"": ""LOGIN_REQUIRED"", ""orders_count"": 0}",PENDING,,,Checkout.place() orders side effect
TC-PRM-001,REQ-07,COND-PRM-01,TS-PRM-INVALID,Từ chối voucher không tồn tại trong hệ thống,P1,"User authenticated, giỏ hàng 600k",600k,"{""voucher"": ""INVALID_CODE_999""}",error,"{""error"": ""INVALID_VOUCHER""}",PENDING,,,Promotions.discount() invalid code
TC-PRM-002,REQ-07,COND-PRM-02,TS-PRM-REUSE,Từ chối voucher đã nằm trong used_codes của khách hàng,P0,"User đã sử dụng SAVE10 (used_codes có SAVE10), giỏ hàng 600k",used_voucher,"{""voucher"": ""SAVE10""}",error,"{""error"": ""VOUCHER_USED""}",PENDING,,,Promotions.discount() reused voucher
TC-PRM-003,REQ-07,COND-PRM-03,TS-PRM-SUCCESS,Đánh dấu voucher vào used_codes khi đặt hàng thành công,P0,"User mới chưa dùng SAVE10, giỏ hàng 600k, COD",600k,"{""voucher"": ""SAVE10"", ""method"": ""COD""}",state,"{""voucher_in_used_codes"": true, ""discount"": 60000}",PENDING,,,Checkout.place() used_codes update
TC-PRM-004,REQ-07,COND-PRM-03,TS-PRM-ROLLBACK,Không đánh dấu voucher used khi thanh toán Online bị Declined,P0,"User mới, giỏ hàng 600k, áp SAVE10, ONLINE declined",600k,"{""voucher"": ""SAVE10"", ""method"": ""ONLINE"", ""approved"": false}",error,"{""error"": ""PAYMENT_DECLINED"", ""voucher_in_used_codes"": false}",PENDING,,,Checkout.place() voucher rollback check
TC-SHP-001,REQ-09,COND-SHP-01,TS-SHP-INNER,Miễn phí ship nội thành đúng tại ngưỡng 500.000 VND (Boundary),P0,"Subtotal đúng 500.000đ, area=inner",500k,"{""area"": ""inner""}",return_value,"{""shipping"": 0}",PENDING,,,Shipping.calculate() 500k inner
TC-SHP-002,REQ-09,COND-SHP-01,TS-SHP-INNER,Phí ship nội thành 30.000 VND ngay dưới ngưỡng 499.999 VND (Boundary),P0,"Subtotal 499.999đ, area=inner",499999,"{""area"": ""inner""}",return_value,"{""shipping"": 30000}",PENDING,,,Shipping.calculate() 499999 inner
TC-SHP-003,REQ-09,COND-SHP-01,TS-SHP-INNER,Miễn phí ship nội thành khi subtotal trên ngưỡng (600.000 VND),P1,"Subtotal 600.000đ, area=inner",600k,"{""area"": ""inner""}",return_value,"{""shipping"": 0}",PENDING,,,Shipping.calculate() 600k inner
TC-SHP-004,REQ-09,COND-SHP-02,TS-SHP-OUTER,Phí ship ngoại thành cố định 50.000 VND dù subtotal >= 500k,P1,"Subtotal 600.000đ, area=outer",600k,"{""area"": ""outer""}",return_value,"{""shipping"": 50000}",PENDING,,,Shipping.calculate() outer area
TC-SHP-005,REQ-09,COND-SHP-03,TS-SHP-INVALID,Từ chối khu vực vận chuyển không hợp lệ (Negative),P2,"Subtotal 600.000đ, area=hanoi_suburb",600k,"{""area"": ""hanoi_suburb""}",error,"{""error"": ""INVALID_AREA""}",PENDING,,,Shipping.calculate() invalid area
TC-PAY-001,REQ-10,COND-PAY-01,TS-PAY-COD,Thanh toán COD thành công tạo Order trạng thái PENDING_COD,P1,"User authenticated, giỏ hàng 600k",600k,"{""method"": ""COD""}",state,"{""order_status"": ""PENDING_COD"", ""orders_count"": 1}",PENDING,,,Payments.collect() COD status
TC-PAY-002,REQ-10,COND-PAY-02,TS-PAY-ONLINE-OK,Thanh toán ONLINE approved tạo Order trạng thái PAID và ghi nhận charge,P0,"User authenticated, giỏ hàng 600k, method ONLINE approved=True",600k,"{""method"": ""ONLINE"", ""approved"": true}",state,"{""order_status"": ""PAID"", ""charges_count"": 1}",PENDING,,,Payments.collect() ONLINE approved
TC-PAY-003,REQ-10,COND-PAY-03,TS-PAY-ONLINE-FAIL,"Thanh toán ONLINE declined bị từ chối, không tạo Order và hoàn trả tồn kho",P0,"User authenticated, giỏ hàng 600k, method ONLINE approved=False",600k,"{""method"": ""ONLINE"", ""approved"": false}",error,"{""error"": ""PAYMENT_DECLINED"", ""orders_count"": 0, ""stock_restored"": true}",PENDING,,,Payments.collect() ONLINE declined


Overwriting STLab02_TestCases.csv


---
## PHẦN 3 - KIẾN TRÚC TEST RUNNER TỰ ĐỘNG (`test_runner.py`)


In [28]:
%%writefile test_runner.py
import sys
if hasattr(sys.stdout, 'reconfigure'):
    try:
        sys.stdout.reconfigure(encoding='utf-8', errors='replace')
    except Exception:
        pass

from pathlib import Path
import csv, datetime, html, json
from collections import Counter
import pandas as pd
from decimal import Decimal
from ecommerce_app import Checkout, Customer, Cart, Product, D

ROOT = Path(__file__).resolve().parent

def fixture(subtotal_case='600k'):
    """Test Fixture: Provides isolated environment for each test case. 
       Old def fixture in ecommerce_app.py doesnt have enough cases for 5 new REQ"""
    service = Checkout()
    customer = Customer('U_NEW', authenticated=True)
    cart = Cart(service.catalog)
    if subtotal_case == 'clean_cart':
        pass  # Fresh empty cart for testing Cart operations
    elif subtotal_case == 'empty_cart':
        pass  # Empty cart
    elif subtotal_case == 'unauth_user':
        customer = Customer('U_GUEST', authenticated=False)
        cart.add('A', 3)
    elif subtotal_case == 'used_voucher':
        customer = Customer('U_USED', authenticated=True, used_codes={'SAVE10'})
        cart.add('A', 3)
    elif subtotal_case == '500k':
        cart.add('A', 1)
        cart.add('B', 1)
    elif subtotal_case == '499999':
        cart.add('A', 1)
        cart.add('B', 1)
        service.catalog.products['B'].price = D(299999)
    elif subtotal_case == '600k':
        cart.add('A', 3)
    elif subtotal_case == '1000k':
        cart.add('A', 5)
    elif subtotal_case == '1500k':
        cart.add('B', 5)
    elif subtotal_case == '300k':
        cart.add('B', 1)
    else:
        raise ValueError(f"UNKNOWN_FIXTURE: {subtotal_case}")
    return service, customer, cart

def execute_case(row):
    app, user, cart = fixture(row['fixture_key'])
    req = row['requirement_id']
    inp = json.loads(row['input_json']) if row.get('input_json') else {}
    exp = json.loads(row['expected_json']) if row.get('expected_json') else {}
    actual_res = {}
    err = ''
    passed = False
    before_stock = {k: p.stock for k, p in app.catalog.products.items()}
    before_orders_count = len(app.orders)
    try:
        if req == 'REQ-02':
            # Cart: Validate addition and line aggregation
            for action in inp.get('actions', []):
                cart.add(action['sku'], action['quantity'])
            actual_res = {
                'total_quantity': sum(line.quantity for line in cart.lines),
                'lines_count': len(cart.lines),
                'subtotal': int(cart.subtotal())
            }
            passed = (actual_res == exp)
        elif req == 'REQ-04':
            # Authentication: Check login enforcement
            order = app.place(user, cart, method=inp.get('method', 'COD'), key=row['test_case_id'])
            actual_res = {
                'status': order.status,
                'orders_count': len(app.orders)
            }
            passed = (actual_res.get('status') == exp.get('status'))
        elif req == 'REQ-07':
            # Promotion: Invalid voucher, reuse, and used_codes updates
            v = inp.get('voucher')
            m = inp.get('method', 'COD')
            appr = inp.get('approved', True)
            order = app.place(user, cart, voucher=v, method=m, approved=appr, key=row['test_case_id'])
            actual_res = {
                'voucher_in_used_codes': (v in user.used_codes),
                'discount': int(order.discount)
            }
            passed = (actual_res == exp)
        elif req == 'REQ-09':
            # Shipping: Calculate shipping fee based on area and subtotal
            subtotal = cart.subtotal()
            area = inp.get('area')
            fee = app.shipping.calculate(subtotal, area)
            actual_res = {'shipping': int(fee)}
            passed = (actual_res == exp)
        elif req == 'REQ-10':
            # Payment: COD status and Online payment approval/decline
            m = inp.get('method')
            appr = inp.get('approved', True)
            order = app.place(user, cart, method=m, approved=appr, key=row['test_case_id'])
            actual_res = {
                'order_status': order.status,
                'orders_count': len(app.orders),
                'charges_count': len(app.payments.charges)
            }
            passed = all(actual_res.get(k) == exp[k] for k in exp)
        else:
            raise ValueError(f"UNKNOWN_REQUIREMENT_TEST: {req}")
    except Exception as e:
        err = str(e)
        after_stock = {k: p.stock for k, p in app.catalog.products.items()}
        stock_restored = (after_stock == before_stock) # Can inventory be returned?
        actual_res = {
            'error': err,
            'orders_count': len(app.orders),
            'voucher_in_used_codes': (inp.get('voucher') in user.used_codes) if inp.get('voucher') else False,
            'stock_restored': stock_restored
        }
        if 'error' in exp:
            passed = (exp['error'] == err)
            if 'orders_count':
                if 'orders_count' in exp:
                    passed = passed and (exp['orders_count'] == len(app.orders))
            if 'voucher_in_used_codes' in exp:
                passed = passed and (exp['voucher_in_used_codes'] == actual_res['voucher_in_used_codes'])
            if 'stock_restored' in exp:
                passed = passed and (exp['stock_restored'] == stock_restored)
        else:
            passed = False
    after_stock = {k: p.stock for k, p in app.catalog.products.items()}
    status = 'PASS' if passed else 'FAIL'
    return {
        **row,
        'execution_status': status,
        'actual_result': json.dumps(actual_res, ensure_ascii=False),
        'error_message': err,
        'stock_before': json.dumps(before_stock),
        'stock_after': json.dumps(after_stock),
        'executed_at': datetime.datetime.now(datetime.timezone.utc).isoformat()
    }

def run_suite(csv_path=None, out_dir=None):
    csv_path = Path(csv_path or ROOT / 'STLab02_TestCases.csv')
    output = Path(out_dir or ROOT / 'reports')
    output.mkdir(exist_ok=True, parents=True)
    with csv_path.open(newline='', encoding='utf-8-sig') as f:
        testcases = list(csv.DictReader(f))
    ids = [x['test_case_id'] for x in testcases]
    if len(ids) != len(set(ids)):
        raise ValueError('DUPLICATE_TEST_IDS detected!')
    results = [execute_case(x) for x in testcases]
    df = pd.DataFrame(results)
    counts = Counter(df['execution_status'])
    summary = df.groupby(['requirement_id', 'execution_status']).size().unstack(fill_value=0).reset_index()

    totals = pd.DataFrame([{
        'executed': len(df),
        'passed': counts['PASS'],
        'failed': counts.get('FAIL', 0),
        'blocked': counts.get('BLOCKED', 0),
        'pass_rate': round(100 * counts['PASS'] / len(df), 1)
    }])
    # 1. Export CSV execution log
    df.to_csv(output / 'test_execution.csv', index=False, encoding='utf-8-sig')
    # 2. Export Excel report with 3 sheets
    try:
        with pd.ExcelWriter(output / 'test_report.xlsx', engine='openpyxl') as xls:
            df.to_excel(xls, index=False, sheet_name='Execution')
            summary.to_excel(xls, index=False, sheet_name='Traceability')
            totals.to_excel(xls, index=False, sheet_name='Summary')
    except Exception:
        pass
    # 3. Export HTML dashboard
    rows = ''.join(
        '<tr>' + ''.join(
            f'<td>{html.escape(str(row.get(key, "")))}</td>'
            for key in ['test_case_id', 'requirement_id', 'condition_id', 'scenario_id', 'expected_json', 'actual_result', 'execution_status']
        ) + '</tr>'
        for row in results
    )
    page = f'''<!DOCTYPE html><html><head><meta charset="utf-8"><title>STLab02 Test Report</title>
<style>
body{{font:15px Arial;margin:30px;color:#1e293b;background:#f8fafc}}
h1{{color:#0f172a}}
.metric-box{{display:flex;gap:15px;margin-bottom:20px}}
.metric{{background:white;padding:15px 25px;border-radius:10px;box-shadow:0 1px 3px rgba(0,0,0,0.1);font-size:18px}}
.pass{{color:#16a34a;font-weight:bold}}
.fail{{color:#dc2626;font-weight:bold}}
table{{border-collapse:collapse;width:100%;background:white;box-shadow:0 1px 3px rgba(0,0,0,0.1);border-radius:8px;overflow:hidden}}
td,th{{border:1px solid #e2e8f0;padding:10px 12px;text-align:left;font-size:14px}}
th{{background:#1e293b;color:white}}
tr:nth-child(even){{background:#f1f5f9}}
</style></head><body>
<h1>STLab02 – EduCommerce Automated Test Execution</h1>
<div class="metric-box">
  <div class="metric">Total: <b>{len(df)}</b></div>
  <div class="metric pass">PASS: <b>{counts["PASS"]}</b></div>
  <div class="metric fail">FAIL: <b>{counts.get("FAIL", 0)}</b></div>
  <div class="metric">Pass Rate: <b>{totals["pass_rate"].iloc[0]}%</b></div>
</div>
<h2>Traceability & Execution Details</h2>
<table>
  <tr><th>Test Case ID</th><th>Requirement</th><th>Condition</th><th>Scenario</th><th>Expected JSON</th><th>Actual Result</th><th>Status</th></tr>
  {rows}
</table>
</body></html>'''
    (output / 'test_dashboard.html').write_text(page, encoding='utf-8')
    print("\n" + "=" * 72)
    print("            STLab02 - EDUCOMMERCE AUTOMATED TEST EXECUTION")
    print("=" * 72)
    print(f"  Executed: {len(df)}  |  PASS: {counts['PASS']}  |  FAIL: {counts.get('FAIL', 0)}  |  Pass Rate: {totals['pass_rate'].iloc[0]}%")
    print("-" * 72)
    if counts.get('FAIL', 0) > 0:
        print("  [FAILED TEST CASES / DEFECTS DETECTED]:")
        for _, f_row in df[df['execution_status'] == 'FAIL'].iterrows():
            print(f"  * Case ID     : {f_row['test_case_id']}")
            print(f"    Traceability: {f_row['requirement_id']} -> {f_row['condition_id']} -> {f_row['scenario_id']}")
            print(f"    Expected    : {f_row['expected_json']}")
            print(f"    Actual      : {f_row['actual_result']}")
            print(f"    Defect ID   : DEF-PAY-01 (Missing inventory rollback on declined payment)")
            print("  " + "-" * 68)
    print(f"  Reports output directory: {output.resolve()}")
    print("=" * 72 + "\n")
    return df, output


if __name__ == '__main__':
    run_suite()


Overwriting test_runner.py


---
## PHẦN 4 - THỰC THI TỰ ĐỘNG BỘ TEST VÀ XUẤT BÁO CÁO

In [29]:
!pip -q install pandas openpyxl matplotlib gradio

import importlib
import test_runner
importlib.reload(test_runner)

from test_runner import run_suite
results_df, report_path = run_suite("STLab02_TestCases.csv", "reports")
print("\n--- BẢNG KẾT QUẢ THỰC THI CHI TIẾT ---")
cols_to_show = ["test_case_id", "requirement_id", "scenario_id", "priority", "execution_status"]
print(results_df[cols_to_show].to_string(index=False))



            STLab02 - EDUCOMMERCE AUTOMATED TEST EXECUTION
  Executed: 19  |  PASS: 18  |  FAIL: 1  |  Pass Rate: 94.7%
------------------------------------------------------------------------
  [FAILED TEST CASES / DEFECTS DETECTED]:
  * Case ID     : TC-PAY-003
    Traceability: REQ-10 -> COND-PAY-03 -> TS-PAY-ONLINE-FAIL
    Expected    : {"error": "PAYMENT_DECLINED", "orders_count": 0, "stock_restored": true}
    Actual      : {"error": "PAYMENT_DECLINED", "orders_count": 0, "voucher_in_used_codes": false, "stock_restored": false}
    Defect ID   : DEF-PAY-01 (Missing inventory rollback on declined payment)
  --------------------------------------------------------------------
  Reports output directory: G:\Download\testSoftware\Lab02\reports


--- BẢNG KẾT QUẢ THỰC THI CHI TIẾT ---
test_case_id requirement_id        scenario_id priority execution_status
  TC-CRT-001         REQ-02       TS-CRT-VALID       P1             PASS
  TC-CRT-002         REQ-02       TS-CRT-VALID       P1

### Bảng Thống kê Traceability Coverage theo Requirement

In [30]:
import pandas as pd
summary_table = results_df.groupby(["requirement_id", "execution_status"]).size().unstack(fill_value=0)
summary_table["TOTAL"] = summary_table.sum(axis=1)
summary_table["PASS_RATE_%"] = (summary_table["PASS"] / summary_table["TOTAL"] * 100).round(1)
print(summary_table)

execution_status  FAIL  PASS  TOTAL  PASS_RATE_%
requirement_id                                  
REQ-02               0     4      4        100.0
REQ-04               0     3      3        100.0
REQ-07               0     4      4        100.0
REQ-09               0     5      5        100.0
REQ-10               1     2      3         66.7


---
## BÁO CÁO DEFECT THỰC TẾ PHÁT HIỆN QUA KIỂM THỬ TỰ ĐỘNG (DEF-PAY-01)
Khi thực thi ca kiểm thử `TC-PAY-003`, hệ thống đã phát hiện chính xác **01 Defect mức độ Critical** trong `ecommerce_app.py`:

| Thuộc tính | Chi tiết Defect |
| :--- | :--- |
| **Defect ID** | **DEF-PAY-01** |
| **Title** | Tồn kho không được hoàn trả (Rollback) khi giao dịch thanh toán Online bị cổng từ chối (`PAYMENT_DECLINED`) |
| **Severity / Priority** | **Critical / P0 (Blocker)** |
| **Vị trí lỗi** | File `ecommerce_app.py` → Class `Checkout`, hàm `place()`, dòng 94–97 |
| **Precondition** | User đăng nhập, giỏ hàng có 3 SKU A (giá 200k/cái, tồn kho ban đầu = 20) |
| **Expected Result** | Bị từ chối với lỗi `PAYMENT_DECLINED`, `orders_count == 0`, tồn kho SKU A phải bảo toàn 20 (`stock_restored == True`) |
| **Actual Result** | Bị từ chối với lỗi `PAYMENT_DECLINED`, không tạo đơn, nhưng tồn kho SKU A bị trừ còn 17 (`stock_restored == False`). Mất trắng 3 sản phẩm |
| **Root Cause** | Lệnh `self.inventory.reserve(cart.lines)` trừ kho trước khi gọi thanh toán. Khi `self.payments.collect()` ném ngoại lệ `PAYMENT_DECLINED`, hàm `place()` không có `try...except` để gọi `self.inventory.release(cart.lines)` hoàn kho. |
| **Code Fix** | Bọc lệnh thanh toán trong `try...except` và gọi `self.inventory.release(cart.lines)` trước khi `raise`. |


---
## PHẦN 5 - TRỰC QUAN HÓA BẰNG GRADIO DASHBOARD

In [31]:
import gradio as gr
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

report_df = pd.read_csv("reports/test_execution.csv").fillna("")

def dashboard_view(req="ALL", status="ALL"):
    df = report_df.copy()
    if req != "ALL":
        df = df[df["requirement_id"] == req]
    if status != "ALL":
        df = df[df["execution_status"] == status]
    
    summary = df["execution_status"].value_counts().reindex(["PASS", "FAIL"], fill_value=0)
    
    fig, ax = plt.subplots(figsize=(5, 3.5))
    bars = ax.bar(summary.index, summary.values, color=["#16a34a", "#dc2626"], width=0.5)
    ax.bar_label(bars, padding=3, fontsize=11, fontweight="bold")
    ax.set_title("Phân bố kết quả kiểm thử (STLab02)", fontsize=13, fontweight="bold")
    ax.set_ylabel("Số lượng test cases")
    ax.set_ylim(0, max(2, int(summary.max()) + 3))
    fig.tight_layout()
    
    cols = ["test_case_id", "requirement_id", "scenario_id", "title", "expected_json", "actual_result", "execution_status"]
    info_text = f"**Tổng số case hiển thị:** {len(df)} | **PASS:** {summary['PASS']} | **FAIL:** {summary['FAIL']}"
    return fig, df[cols], info_text

with gr.Blocks(title="STLab02 - EduCommerce Test Dashboard") as demo:
    gr.Markdown("# EduCommerce — Bảng điều khiển kết quả kiểm thử (STLab02)")
    with gr.Row():
        req_filter = gr.Dropdown(["ALL"] + sorted(report_df["requirement_id"].unique().tolist()), value="ALL", label="Lọc theo Requirement")
        status_filter = gr.Dropdown(["ALL", "PASS", "FAIL"], value="ALL", label="Lọc theo Trạng thái")
    
    status_line = gr.Markdown()
    with gr.Row(equal_height=True):
        with gr.Column(scale=1, min_width=320):
            chart = gr.Plot(label="Biểu đồ phân bố")
        with gr.Column(scale=2, min_width=500):
            result_table = gr.Dataframe(label="Chi tiết Test Cases", interactive=False, wrap=True)
            
    gr.File(value="reports/test_report.xlsx", label="Tải xuống Báo cáo Excel (.xlsx)")
    
    req_filter.change(dashboard_view, [req_filter, status_filter], [chart, result_table, status_line])
    status_filter.change(dashboard_view, [req_filter, status_filter], [chart, result_table, status_line])
    demo.load(dashboard_view, [req_filter, status_filter], [chart, result_table, status_line])

demo.launch(share=False, debug=False)

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


### Tải file kết quả về máy (Colab)

In [32]:
try:
    from google.colab import files
    files.download("reports/test_report.xlsx")
    files.download("reports/test_execution.csv")
    files.download("reports/test_dashboard.html")
except ImportError:
    print("Môi trường local: Các file báo cáo đã nằm trong thư mục 'reports/'.")

Môi trường local: Các file báo cáo đã nằm trong thư mục 'reports/'.
